In [1]:
#check on December-8, 2025 by Hengcong Liu

import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc
import xgboost as xgb

# ==============================
# 预加载静态数据
# ==============================
risk_factor_df = pd.read_excel('output/1_risk_factor/risk_factor.xlsx')
weight_df = pd.read_excel("output/2_weight/weight.xlsx")[["gb", "rescaled"]]
data = pd.read_excel("../1_clean/data_merge_all_20260110.xlsx")
data = data[~(data['host_species'].isna() & data['standard virus name'].isna())].reset_index(drop=True)
print(data.shape[0])

# baseline 的 case_control_df
# 要生成的列名及对应的分类字段
host_levels = {
    "species": "host_species",
    "genus": "host_genus"
}
case_control_dfs = {}  # 用字典存储结果
for level, col in host_levels.items():
    case_control_dfs[level] = (
        data.assign(presence=1)
            .pivot_table(
                index="gb",
                columns=col,
                values="presence",
                aggfunc="max",
                fill_value=0
            )
            .reset_index()
    )

# ===========================
#       文件保存辅助函数
# ===========================
def save_ci(values, target_species, outname):
    df = pd.DataFrame({
        "index": ["mean", "lbd", "ubd"],
        "value": [np.mean(values), np.percentile(values, 2.5), np.percentile(values, 97.5)]
    })
    outdir = Path(f"output/4_predict/host/{outname}")
    outdir.mkdir(parents=True, exist_ok=True)
    filename = f"{target_species}.xlsx"
    df.to_excel(outdir / filename, index=False)

def save_df(df, target_species,outdir_name):
    outdir = Path(f"output/4_predict/host/{outdir_name}")
    outdir.mkdir(parents=True, exist_ok=True)
    filename = f"{target_species}.xlsx"
    df.to_excel(outdir / filename, index=False)

# ===========================
#         BRT 主函数
# ===========================
def ran_BRT(target_species,
            learning_rate,
            max_depth,
            n_estimators,
            subsample,
            risk_factor_df,
            weight_df,
            case_control_df):

    all_risk_cols = risk_factor_df.columns.drop("gb")
    all_data = (
        risk_factor_df
        .merge(weight_df, on="gb", how="left")
        .merge(case_control_df[["gb", target_species]], on="gb", how="left")
        .rename(columns={target_species: "status"})
    )
    survey_data = all_data.dropna(subset=["status"]).reset_index(drop=True)
    all_x = all_data[all_risk_cols].values
    all_gb = all_data["gb"].values

    importance_dict = {k: [] for k in all_risk_cols}
    auc_list, tpr_list, tnr_list = [], [], []
    preds_list = []

    xgb_params = dict(
        learning_rate=float(learning_rate),
        max_depth=int(max_depth),
        subsample=float(subsample),
        n_estimators=int(n_estimators)
    )

    for i in range(100):
        train, test = train_test_split(
            survey_data,
            test_size=0.25,
            random_state=i, #固定随机种子，保证结果的可重复性
            stratify=survey_data["status"], #Stratified sampling, 分层抽样，让训练集和测试集按照某一列保持相同的类别比例，可以避免正负样本比例失衡，出现训练集全是0或者没有0的情况
            shuffle=True #在划分前打乱数据的顺序，避免数据被分成两个不同使其，避免顺序性带来的偏差，提高泛化性
        )
        train_x = train[all_risk_cols].values
        train_y = train["status"].values
        train_w = train["rescaled"].values
        scale_pos_weight = sum(train_y == 0) / sum(train_y == 1)

        model = xgb.XGBClassifier(**xgb_params, 
                                  random_state=i, 
                                  scale_pos_weight=scale_pos_weight,
                                    objective="binary:logistic",  #二分类逻辑回归，输出0-1的概率，使用logistic loss（对数损失）
                                  eval_metric="logloss", #评估指标，对数损失
                                  tree_method="hist", #推荐的tree_method，提升运行速度
                                  n_jobs=max(1, int(os.cpu_count() * 0.7)), #cpu的线程数，1=单线程，-1=自动使用全部cpu，
                                  )
        model.fit(train_x, train_y, sample_weight=train_w)

        # Feature importance
        for idx, k in enumerate(all_risk_cols):
            importance_dict[k].append(model.feature_importances_[idx])

        # ROC
        test_pred = model.predict_proba(test[all_risk_cols].values)[:, 1]
        fpr, tpr, th = roc_curve(test["status"], test_pred)
        best_idx = np.argmax(tpr - fpr)
        auc_list.append(auc(fpr, tpr))
        tpr_list.append(tpr[best_idx])
        tnr_list.append(1 - fpr[best_idx])
        best_threshold = th[best_idx]

        all_pred = model.predict_proba(all_x)[:, 1]
        preds_list.append(pd.DataFrame({
            "round": i,
            "threshold": best_threshold,
            "gb": all_gb,
            "pred": all_pred
        }))

    # 保存结果
    save_ci(auc_list, target_species, "auc")
    save_ci(tpr_list, target_species, "tpr")
    save_ci(tnr_list, target_species, "tnr")

    rel_con = pd.DataFrame(importance_dict).T.reset_index().rename(columns={"index": "Key"})
    rel_con["row_sum"] = rel_con.iloc[:, 1:].sum(axis=1)
    rel_con["std_sum"] = rel_con["row_sum"] / rel_con["row_sum"].sum()
    save_df(rel_con, target_species, "rc")

    pred_df = pd.concat(preds_list, ignore_index=True)
    save_df(pred_df, target_species, "prob")

    print(f"✓ BRT finished → {target_species}")

# ===========================
#       批量运行
# ===========================
grid = pd.read_excel("output/3_grid_search/host_best_params.xlsx")

for _, row in grid.iterrows():
    target_species = row["species"]
    learning_rate = row["learning_rate"]
    max_depth = row["max_depth"]
    n_estimators = row["n_estimators"]
    subsample = row["subsample"]
    if target_species=='Rattus' or target_species=='Crocidura':
        case_control_df=case_control_dfs['genus']
    else:
        case_control_df=case_control_dfs['species']
    ran_BRT(target_species, learning_rate, max_depth, n_estimators, subsample, risk_factor_df=risk_factor_df, weight_df=weight_df, case_control_df=case_control_df)

47196
✓ BRT finished → Apodemus agrarius
✓ BRT finished → Rattus norvegicus
✓ BRT finished → Suncus murinus
✓ BRT finished → Crocidura
✓ BRT finished → Rattus
